In [1]:
import os
import sys
import logging
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.append(project_root)
from config.conf import create_spark_session
from config.conf import config
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [2]:
spark = create_spark_session()

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [4]:
try:
    input_path = f"{config.root_path}{config.bronze_zone}transfers"
    output_path = f"{config.root_path}{config.silver_zone}transfers"

    logger.info(f"Lecture du fichier transfers depuis : {input_path}")
    df_input = spark.read.option("header", "true").parquet(input_path)

    df = df_input \
        .withColumn("transfer_fee", F.col("transfer_fee").cast("int")) \
        .withColumn("market_value_in_eur", F.col("market_value_in_eur").cast("int")) \
        .withColumn("price_difference", F.col("transfer_fee") - F.col("market_value_in_eur")) \
        .withColumn("transfer_year", F.year("transfer_date"))

    logger.info(f"Écriture du DataFrame transformé au format Parquet vers : {output_path}")
    df.write.mode("overwrite").parquet(output_path)

    logger.info("✅ Traitement de transfers terminé avec succès.\n")

except Exception as e:
    logger.error(f"❌ Erreur lors du traitement de transfers : {e}", exc_info=True)

2025-05-19 09:38:54,031 - INFO - Lecture du fichier transfers depuis : s3a://loicverdier/bronze/football_transfermarkt/transfers
2025-05-19 09:39:00,521 - INFO - Écriture du DataFrame transformé au format Parquet vers : s3a://loicverdier/silver/football_transfermarkt/transfers
2025-05-19 09:39:10,629 - INFO - ✅ Traitement de transfers terminé avec succès. 

